# Notebook 01 - Generate and Load Operational Source Data

## Objective

Simulate the operational source systems used by **Verdanova Consumer Products Ltd.** and ingest the generated data into the Microsoft Fabric Lakehouse.

This notebook generates realistic synthetic business data for each operational system, stores the raw exports in the **Lakehouse Files** area, and loads them into the **Bronze layer** as Delta tables without applying any transformations.

## Business Context

Verdanova Consumer Products Ltd. operates multiple business systems, including:

* Customer Relationship Management (CRM)
* Enterprise Resource Planning (ERP)
* Marketing Campaign Platform
* Excel-based Planning

These operational systems produce the raw business data used for analytics. This notebook simulates their daily exports and establishes the initial data foundation for the Medallion architecture.

## Pipeline Position

This is **Notebook 01** in the Medallion architecture. It generates the raw operational data, stores the exports in the Lakehouse **Files** area, and loads them into **Bronze Delta tables**.

The generated Files and Bronze tables are used by the downstream profiling, Silver transformation, and Gold modeling notebooks.

## Process

For each operational system, this notebook performs the following steps:

1. Generate realistic synthetic operational data.
2. Store the raw data in the **Lakehouse Files** area.
3. Load the raw data into **Bronze Delta tables**.

No data cleansing, standardization, or business transformations are performed in this notebook.

## Expected Output

### Lakehouse Files

```text
Files/
├── CRM/
├── ERP/
├── Marketing/
└── Excel/
```

### Bronze Tables

* bronze_customers
* bronze_products
* bronze_orders
* bronze_order_lines
* bronze_campaign_performance
* bronze_sales_targets

This notebook establishes the raw operational data foundation for subsequent profiling, transformation, and analytical modeling notebooks.


# Section 1 - Customer Relationship Management (CRM)

## Objective
Simulate CRM operational export containing customer demographic and registration data, including intentional source-system quality defects.

### CRM Data Generation
Generates synthetic customer records with realistic data quality defects (null values, inconsistent casing, duplicate rows) to validate downstream Bronze profiling and Silver deduplication/cleansing logic.

In [3]:
from pyspark.sql import SparkSession
import random
from datetime import datetime, timedelta

spark = SparkSession.builder.getOrCreate()

regions = ["North","South","East","West"]
customer_types = ["Retail","Wholesale","Distributor"]

customers = []

for i in range(1,1001):

    join_date = datetime(2024,1,1) + timedelta(days=random.randint(0,730))

    customers.append({
        "CustomerID":10000+i,
        "CustomerName":f"Customer {i}",
        "CustomerType":random.choice(customer_types),
        "Region":random.choice(regions),
        "Email":f"customer{i}@example.com",
        "JoinDate":join_date.strftime("%Y-%m-%d")
    })

# Intentional data quality issues

customers[10]["Email"] = None
customers[25]["Region"] = "NORTH"
customers.append(customers[50])
customers.append(customers[120])

df = spark.createDataFrame(customers)

display(df)

StatementMeta(, 40435ccd-2e94-4b47-90cb-1b62e5c7f624, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6f17b970-5c43-48f3-829c-4fce3e115d50)

**Save to Lakehouse Files**

In [4]:
output_path = "Files/CRM/Customers"

(
    df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header",True)
    .csv(output_path)
)

print("CRM customer data exported successfully.")

StatementMeta(, 40435ccd-2e94-4b47-90cb-1b62e5c7f624, 6, Finished, Available, Finished, False)

CRM customer data exported successfully.


**Load the raw CRM customer data from the Lakehouse **Files** area into the **Bronze layer** as a Delta table. The Bronze layer preserves the original operational data without applying any cleansing, standardization, or business transformations.**

In [5]:
# Read CRM CSV files

crm_df = (
    spark.read
         .option("header", True)
         .csv("Files/CRM/Customers")
)

display(crm_df)

StatementMeta(, 40435ccd-2e94-4b47-90cb-1b62e5c7f624, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fb022655-2012-4e24-b029-77b01526cc55)

In [7]:
(
    crm_df.write
          .format("delta")
          .mode("overwrite")
          .saveAsTable("bronze_customers")
)

StatementMeta(, 40435ccd-2e94-4b47-90cb-1b62e5c7f624, 9, Finished, Available, Finished, False)

## ERP Products, Orders, and Order Lines

This section generates the ERP product master, customer order, and order line data, exports the raw data to the Lakehouse Files area, and loads the corresponding Bronze Delta tables.

### Products
Generates product master records with cost and pricing information and loads them into `bronze_products`.

### Orders
Generates customer order transactions and loads them into `bronze_orders`.

### Order Lines
Generates the individual product lines associated with each order and loads them into `bronze_order_lines`.

**Generate Products**

In [1]:
from pyspark.sql import SparkSession
import random

spark = SparkSession.builder.getOrCreate()

categories = [
    "Drinkware",
    "Kitchen",
    "Cleaning",
    "Office"
]

products = []

for i in range(1, 51):

    # Generate a realistic cost
    standard_cost = round(random.uniform(5, 80), 2)

    # Generate selling price from cost using a controlled 25%-60% markup
    margin = random.uniform(0.25, 0.60)
    unit_price = round(standard_cost * (1 + margin), 2)

    products.append({
        "ProductID": i,
        "ProductName": f"Product {i}",
        "Category": random.choice(categories),
        "StandardCost": standard_cost,
        "UnitPrice": unit_price
    })

# Intentional quality issues for profiling
products[8]["Category"] = None
products.append(products[20])

products_df = spark.createDataFrame(products)

display(products_df)

StatementMeta(, a3c96f56-5178-4941-92b8-c11d9379ab07, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1bd80d07-1ef0-4b3e-a1b1-44862cd6a4e4)

**Save Products to Files**

In [2]:
products_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("Files/ERP/Products")

print("Products exported successfully.")

StatementMeta(, a3c96f56-5178-4941-92b8-c11d9379ab07, 4, Finished, Available, Finished, False)

Products exported successfully.


**Load Products to Bronze**

In [3]:
products_bronze = spark.read \
    .option("header", True) \
    .csv("Files/ERP/Products")

products_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_products")

display(products_bronze)

StatementMeta(, a3c96f56-5178-4941-92b8-c11d9379ab07, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 63ceb489-8861-4b2d-ac22-42c0ce5c86bd)

### ERP Orders Ingestion
Simulates sales order headers across Retail, Wholesale, and Online channels for 2025 and ingests them into the raw `bronze_orders` Delta table.

In [11]:
from datetime import datetime, timedelta

orders = []

for i in range(1, 1001):

    order_date = datetime(2025,1,1) + timedelta(days=random.randint(0,365))

    orders.append({

        "OrderID": i,
        "CustomerID": random.randint(10001,11000),
        "OrderDate": order_date.strftime("%Y-%m-%d"),
        "Channel": random.choice([
            "Retail",
            "Wholesale",
            "Online"
        ])
    })

orders_df = spark.createDataFrame(orders)

display(orders_df)

StatementMeta(, 40435ccd-2e94-4b47-90cb-1b62e5c7f624, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d31bee8a-1639-446c-a329-69d06324c658)

**Save & Load Orders**

In [12]:
orders_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("Files/ERP/Orders")

spark.read.option("header",True).csv("Files/ERP/Orders") \
    .write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_orders")

StatementMeta(, 40435ccd-2e94-4b47-90cb-1b62e5c7f624, 14, Finished, Available, Finished, False)

### ERP Order Lines Ingestion
Generates transactional order line items by looking up product prices and costs from `bronze_products`. Exports raw CSV files and writes the `bronze_order_lines` Delta table.

In [1]:
from pyspark.sql import SparkSession
import random

spark = SparkSession.builder.getOrCreate()

# Get product master data
products = spark.table("bronze_products").collect()

product_lookup = {
    int(row["ProductID"]): {
        "UnitPrice": float(row["UnitPrice"]),
        "StandardCost": float(row["StandardCost"])
    }
    for row in products
}

order_lines = []
line_id = 1

for order in range(1, 1001):

    for _ in range(random.randint(1, 4)):

        product_id = random.randint(1, 50)

        quantity = random.randint(1, 5)

        unit_price = product_lookup[product_id]["UnitPrice"]
        cost = product_lookup[product_id]["StandardCost"]

        discount = round(random.uniform(0, 0.20), 2)

        line_amount = round(
            quantity * unit_price * (1 - discount),
            2
        )

        order_lines.append({

            "OrderLineID": line_id,
            "OrderID": order,
            "ProductID": product_id,
            "Quantity": quantity,
            "UnitPrice": unit_price,
            "CostAtSale": cost,
            "Discount": discount,
            "LineAmount": line_amount

        })

        line_id += 1

order_lines_df = spark.createDataFrame(order_lines)

order_lines_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("Files/ERP/OrderLines")

spark.read.option("header", True).csv("Files/ERP/OrderLines") \
    .write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_order_lines")

display(spark.table("bronze_order_lines"))

StatementMeta(, ce39cef3-12d9-4cbf-9391-47d2a2cbae8d, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9c351ce3-9c46-4b3a-a981-ed22bf045f26)

# Section 3 - Marketing Campaign Platform

## Objective
Simulate marketing campaign metrics across digital ad platforms. Revenue attributed to campaigns is dynamically scaled as a 60% envelope of actual 2025 sales to maintain realistic business proportions.

*Prerequisite Dependency: Code cell [2] queries `spark.table("sales")` to compute the total revenue target. Ensure `sales` exists as a table or view (e.g., aliased from `bronze_order_lines`) in the Lakehouse before executing this section.*

In [2]:
from pyspark.sql import SparkSession
import random
from datetime import datetime, timedelta

spark = SparkSession.builder.getOrCreate()

random.seed(42)

platforms = [
    "Facebook",
    "Instagram",
    "LinkedIn",
    "Google Ads"
]

regions = [
    "North",
    "South",
    "East",
    "West"
]

categories = [
    "Drinkware",
    "Kitchen",
    "Cleaning",
    "Office"
]


# ---------------------------------------------------------
# 1. Read actual 2025 sales
# ---------------------------------------------------------

sales = spark.table("sales")

sales_2025 = (
    sales
    .filter(sales["OrderDate"].cast("date") >= "2025-01-01")
    .filter(sales["OrderDate"].cast("date") <= "2025-12-31")
)

actual_sales = (
    sales_2025
    .agg({"LineAmount": "sum"})
    .collect()[0][0]
)

print(f"Actual 2025 Sales: {actual_sales:,.2f}")


# ---------------------------------------------------------
# 2. Set marketing revenue envelope
# ---------------------------------------------------------

# Marketing-attributed revenue is intentionally a portion
# of total company sales, not additional company sales.

marketing_revenue_target = round(
    float(actual_sales) * 0.60,
    2
)

print(
    f"Marketing Revenue Target: "
    f"{marketing_revenue_target:,.2f}"
)


# ---------------------------------------------------------
# 3. Generate campaign records
# ---------------------------------------------------------

campaigns = []

raw_campaigns = []

for i in range(1, 301):

    campaign_date = (
        datetime(2025, 1, 1)
        + timedelta(days=random.randint(0, 364))
    )

    impressions = random.randint(5000, 50000)

    clicks = random.randint(200, 5000)

    conversions = random.randint(
        20,
        min(400, clicks)
    )

    spend = round(
        random.uniform(100, 3000),
        2
    )

    raw_campaigns.append({
        "CampaignID": i,
        "CampaignName": f"Campaign {i}",
        "Platform": random.choice(platforms),
        "Region": random.choice(regions),
        "ProductCategory": random.choice(categories),
        "CampaignDate": campaign_date.strftime("%Y-%m-%d"),
        "Impressions": impressions,
        "Clicks": clicks,
        "Conversions": conversions,
        "Spend": spend
    })


# ---------------------------------------------------------
# 4. Generate revenue weights
# ---------------------------------------------------------

weights = [
    random.uniform(0.5, 1.5)
    for _ in raw_campaigns
]

weight_total = sum(weights)


# ---------------------------------------------------------
# 5. Allocate marketing revenue
# ---------------------------------------------------------

for campaign, weight in zip(
    raw_campaigns,
    weights
):

    revenue = round(
        marketing_revenue_target
        * (weight / weight_total),
        2
    )

    campaign["RevenueGenerated"] = revenue

    campaigns.append(campaign)


# ---------------------------------------------------------
# 6. Intentional data-quality issues
# ---------------------------------------------------------

# Missing platform
campaigns[10]["Platform"] = None

# Inconsistent category casing
campaigns[20]["ProductCategory"] = "drinkware"

# Duplicate campaign
campaigns.append(campaigns[30].copy())


# ---------------------------------------------------------
# 7. Create DataFrame
# ---------------------------------------------------------

campaign_df = spark.createDataFrame(campaigns)

display(campaign_df)


# ---------------------------------------------------------
# 8. Write raw campaign files
# ---------------------------------------------------------

campaign_df.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(
        "Files/Marketing/CampaignPerformance"
    )


# ---------------------------------------------------------
# 9. Load Bronze
# ---------------------------------------------------------

spark.read \
    .option("header", True) \
    .csv(
        "Files/Marketing/CampaignPerformance"
    ) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "bronze_campaign_performance"
    )


# ---------------------------------------------------------
# 10. Verify Bronze
# ---------------------------------------------------------

display(
    spark.table(
        "bronze_campaign_performance"
    )
)

print(
    "Bronze campaign rows:",
    spark.table(
        "bronze_campaign_performance"
    ).count()
)

print(
    "Bronze marketing revenue:",
    spark.table(
        "bronze_campaign_performance"
    )
    .agg({
        "RevenueGenerated": "sum"
    })
    .collect()[0][0]
)

StatementMeta(, 01b80035-7c6d-46a5-99fa-05af4cde8726, 4, Finished, Available, Finished, False)

Actual 2025 Sales: 448,878.32
Marketing Revenue Target: 269,326.99


SynapseWidget(Synapse.DataFrame, ce92f092-94d4-447d-8de6-a525025cf632)

SynapseWidget(Synapse.DataFrame, a394f465-6ff3-42d7-b10d-370c5de5dce6)

Bronze campaign rows: 301
Bronze marketing revenue: 269788.70000000007


# Section 4 - Excel Planning (Sales Targets)

## Objective
Simulate annual planning data managed via Excel spreadsheets. Generates seasonal monthly sales targets ($600,000 total) across regions and product categories, exports CSV files to `Files/Excel/SalesTargets`, and loads the `bronze_sales_targets` Delta table.

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

# =========================================================
# VERDANOVA - 2025 SALES TARGET GENERATION
# =========================================================

YEAR = 2025
ANNUAL_TARGET = 600000.00

regions = ["North", "South", "East", "West"]

categories = [
    "Drinkware",
    "Kitchen",
    "Cleaning",
    "Office"
]


# ---------------------------------------------------------
# Planned monthly seasonality
# These are relative weights, not percentages.
# They will be normalized automatically.
# ---------------------------------------------------------

monthly_weights = {
    1: 75,
    2: 70,
    3: 75,
    4: 80,
    5: 85,
    6: 80,
    7: 85,
    8: 90,
    9: 85,
    10: 85,
    11: 85,
    12: 100
}


# ---------------------------------------------------------
# Regional distribution
# ---------------------------------------------------------

region_weights = {
    "North": 1,
    "South": 1,
    "East": 1,
    "West": 1
}


# ---------------------------------------------------------
# Category distribution
# ---------------------------------------------------------

category_weights = {
    "Drinkware": 1,
    "Kitchen": 1,
    "Cleaning": 1,
    "Office": 1
}


# ---------------------------------------------------------
# Normalize weights
# ---------------------------------------------------------

monthly_total_weight = sum(monthly_weights.values())
region_total_weight = sum(region_weights.values())
category_total_weight = sum(category_weights.values())


monthly_share = {
    month: weight / monthly_total_weight
    for month, weight in monthly_weights.items()
}

region_share = {
    region: weight / region_total_weight
    for region, weight in region_weights.items()
}

category_share = {
    category: weight / category_total_weight
    for category, weight in category_weights.items()
}


# ---------------------------------------------------------
# Generate targets
# ---------------------------------------------------------

targets = []

target_id = 1

for month in range(1, 13):

    for region in regions:

        for category in categories:

            target_value = (
                ANNUAL_TARGET
                * monthly_share[month]
                * region_share[region]
                * category_share[category]
            )

            targets.append({
                "TargetID": target_id,
                "Year": YEAR,
                "Month": month,
                "Region": region,
                "ProductCategory": category,
                "SalesTarget": round(target_value, 2),
                "YearMonthKey": YEAR * 100 + month
            })

            target_id += 1


# ---------------------------------------------------------
# Create DataFrame
# ---------------------------------------------------------

targets_df = spark.createDataFrame(targets)


# ---------------------------------------------------------
# Validate
# ---------------------------------------------------------

print("Rows:", targets_df.count())

total_target = (
    targets_df
    .agg(F.sum("SalesTarget").alias("TotalTarget"))
    .first()["TotalTarget"]
)

print(f"Generated target: {total_target:,.2f}")


# ---------------------------------------------------------
# Display monthly targets
# ---------------------------------------------------------

display(
    targets_df
    .groupBy("Year", "Month")
    .agg(
        F.sum("SalesTarget").alias("MonthlyTarget")
    )
    .orderBy("Year", "Month")
)


# ---------------------------------------------------------
# Display complete target
# ---------------------------------------------------------

display(
    targets_df
    .orderBy(
        "Year",
        "Month",
        "Region",
        "ProductCategory"
    )
)


# ---------------------------------------------------------
# Write operational CSV
# ---------------------------------------------------------

(
    targets_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv("Files/Excel/SalesTargets")
)


# ---------------------------------------------------------
# Load Bronze
# ---------------------------------------------------------

(
    spark.read
    .option("header", True)
    .csv("Files/Excel/SalesTargets")
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_sales_targets")
)


# ---------------------------------------------------------
# Verify Bronze
# ---------------------------------------------------------

bronze_df = spark.table("bronze_sales_targets")

print("Bronze rows:", bronze_df.count())

bronze_df.agg(
    F.sum("SalesTarget").alias("TotalSalesTarget")
).show()

StatementMeta(, fbbeb90c-6044-43f3-9a3d-2585448a62ec, 5, Finished, Available, Finished, False)

Rows: 192
Generated target: 600,000.16


SynapseWidget(Synapse.DataFrame, f882f495-50d6-4b17-9aa9-29933d70095c)

SynapseWidget(Synapse.DataFrame, f7c4b2f7-d777-435a-a947-88038fffea76)

Bronze rows: 192
+-----------------+
| TotalSalesTarget|
+-----------------+
|600000.1600000007|
+-----------------+

